# OpenAI sparse-circuit architecture and activation recording

Install `.[openai,notebooks]`. Uses the official circuit_sparsity GPT and hook recorder with tiny random weights and activation sparsity. This is not a pretrained circuit or a pruning reproduction. Released model artifacts can be loaded upstream separately after license and resource review.

Run all cells in a fresh kernel. The final cell records local runtime and kernel RSS.

In [1]:
import os, time, json, resource, sys
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["USE_TF"] = "0"
os.environ["WANDB_MODE"] = "disabled"
import torch, numpy as np
torch.set_num_threads(2)
torch.set_num_interop_threads(2)
torch.manual_seed(7)
np.random.seed(7)
started = time.perf_counter()
print("CPU, fixed seed 7; tiny synthetic fixtures; no weight downloads")


CPU, fixed seed 7; tiny synthetic fixtures; no weight downloads


In [2]:
from autoexplain.frontier import tiny_sparse_circuit_model, inspect_sparse_circuit
model = tiny_sparse_circuit_model()
report = inspect_sparse_circuit(model, torch.tensor([[1,3,4,5]]))
print("Logits:", tuple(report["logits"].shape))
print("Recorded sites:", list(report["activations"]))
print("Parameter zero fraction:", report["parameter_zero_fraction"])
for name, value in list(report["activations"].items())[:8]:
    if isinstance(value, torch.Tensor):
        print(name, tuple(value.shape), "zero fraction", float((value==0).float().mean()))
assert report["activations"] and report["logits"].isfinite().all()


number of parameters: 0.00M
Logits: (1, 4, 16)
Recorded sites: ['0.resid_in', '0.attn.act_in', '0.attn.k', '0.attn.q', '0.attn.v', '0.attn.y', '0.attn.resid_delta', '0.resid_mid', '0.mlp.act_in', '0.mlp.post_act', '0.mlp.resid_delta', '1.resid_in', '1.attn.act_in', '1.attn.k', '1.attn.q', '1.attn.v', '1.attn.y', '1.attn.resid_delta', '1.resid_mid', '1.mlp.act_in', '1.mlp.post_act', '1.mlp.resid_delta', 'final_resid']
Parameter zero fraction: 0.10326086956521739
0.resid_in (1, 4, 8) zero fraction 0.0
0.attn.act_in (1, 4, 8) zero fraction 0.5
0.attn.k (1, 4, 8) zero fraction 0.0
0.attn.q (1, 4, 8) zero fraction 0.0
0.attn.v (1, 4, 8) zero fraction 0.0
0.attn.y (1, 4, 8) zero fraction 0.0
0.attn.resid_delta (1, 4, 8) zero fraction 0.5
0.resid_mid (1, 4, 8) zero fraction 0.0


In [3]:
elapsed = time.perf_counter() - started
rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
rss_mib = rss / (1024**2 if sys.platform == "darwin" else 1024)
print(json.dumps({"tutorial_compute_wall_seconds": round(elapsed, 3),
                 "kernel_lifetime_peak_rss_mib": round(rss_mib, 2),
                 "memory_scope": "kernel only, including imports; not aggregate system memory"}))


{"tutorial_compute_wall_seconds": 0.034, "kernel_lifetime_peak_rss_mib": 276.82, "memory_scope": "kernel only, including imports; not aggregate system memory"}
